<a href="https://colab.research.google.com/github/Ezgicode/FlyRAnk-AI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ezgicode/FlyRAnk-AI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For this data-contract exercise, one raw row represents the daily search and analytics performance of one pseudonymized content page for one pseudonymized client on one report date.

I will work with the `fact_content_daily_performance` table and use the mid-panel partition `month = 2026-03`, covering March 1 to March 31, 2026. This month is used for safe iteration because it is not the final month in the release.

The final project output will later aggregate daily records into one page-level review candidate at a decision point. However, this contract begins with the warehouse table's real grain: one client-content-page-day record.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature fields

The feature frame will use only signals aggregated from the feature window, March 1-15:

- `prior_15d_impressions`
- `prior_15d_clicks`
- `prior_15d_avg_position`
- `prior_15d_ctr`
- `prior_15d_active_days`

These fields are calculated only from information available before the March 15 decision point.

### Label or proxy

The contract-only target will be `is_next_15d_declining`.

A page will receive this proxy label when it has enough impressions in the first 15 days of March and its impressions in the following period, March 16-31, are at least 20 percent lower. This is an observed later-window outcome, not a feature.

### Context fields

`client_hash_id` and `content_hash_id` are pseudonymized identifiers. They are used for grouping, joining, and later validation, but never as model features. `report_date`, `month`, and `ga4_data_available` are also context fields used to define time windows and data availability.

### Excluded fields

All measurements from March 16-31 are excluded from the feature set because they belong to the later outcome window. A label-derived copy, such as the future-impression decline value, will be added only for the deliberate leakage demonstration and then removed. Raw client names, URLs, titles, and private queries are not used or published.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
%pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN secret was not found. Check the Colab Secrets panel.")

con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

# Verification query 1: grain check
grain_check = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS duplicate_count
FROM {MARCH}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 10
"""

print("QUERY 1 - Grain check: duplicate client-content-date rows")
display(con.sql(grain_check).df())

# Verification query 2: row count and date span
count_and_window = f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_hash_id) AS client_count,
    COUNT(DISTINCT content_hash_id) AS content_count,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM {MARCH}
"""

print("QUERY 2 - March 2026 row count and date span")
display(con.sql(count_and_window).df())

# Verification query 3: GA4 availability, explicitly using IS TRUE
availability_check = f"""
SELECT
    COUNT(*) AS ga4_available_rows,
    COUNT(DISTINCT client_hash_id) AS clients_with_ga4_rows,
    COUNT(DISTINCT content_hash_id) AS content_pages_with_ga4_rows
FROM {MARCH}
WHERE ga4_data_available IS TRUE
"""

print("QUERY 3 - Rows surviving ga4_data_available IS TRUE")
display(con.sql(availability_check).df())

QUERY 1 - Grain check: duplicate client-content-date rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,duplicate_count


QUERY 2 - March 2026 row count and date span


,row_count,client_count,content_count,first_report_date,last_report_date
0,9841378,55,331437,2026-03-01,2026-03-31


QUERY 3 - Rows surviving ga4_data_available IS TRUE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_available_rows,clients_with_ga4_rows,content_pages_with_ga4_rows
0,413966,41,90489


### Five-feature frame

The feature frame aggregates the March 1-15 feature window to one client-content page row. The later March 16-31 period is reserved for the decline proxy label.

- `prior_15d_impressions` is knowable at the decision moment because it is measured before March 15.
- `prior_15d_clicks` is knowable at the decision moment because it is measured before March 15.
- `prior_15d_avg_position` is knowable at the decision moment because it is calculated only from the earlier feature window.
- `prior_15d_ctr` is knowable at the decision moment because it is derived from earlier clicks and impressions.
- `prior_15d_active_days` is knowable at the decision moment because it counts earlier days with observed impressions.

The later-window impression total and `is_next_15d_declining` are label information, not features.

In [ ]:
feature_frame_query = f"""
WITH page_windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS prior_15d_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS prior_15d_clicks,

        AVG(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                THEN NULLIF(gsc_avg_position, 0)
            END
        ) AS prior_15d_avg_position,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                     AND gsc_impressions > 0
                THEN 1
                ELSE 0
            END
        ) AS prior_15d_active_days,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS next_16d_impressions

    FROM {MARCH}
    GROUP BY 1, 2
),

scored_frame AS (
    SELECT
        client_hash_id,
        content_hash_id,
        prior_15d_impressions,
        prior_15d_clicks,
        prior_15d_avg_position,
        100.0 * prior_15d_clicks
            / NULLIF(prior_15d_impressions, 0) AS prior_15d_ctr,
        prior_15d_active_days,
        next_16d_impressions,

        CASE
            WHEN prior_15d_impressions >= 100
             AND next_16d_impressions <= 0.80 * prior_15d_impressions
            THEN 1
            ELSE 0
        END AS is_next_15d_declining

    FROM page_windows
    WHERE prior_15d_impressions >= 100
)

SELECT * FROM scored_frame
"""

feature_frame = con.sql(feature_frame_query).df()

print(f"Feature-frame rows: {len(feature_frame):,}")
print(
    "Later-window decline proxy rate: "
    f"{feature_frame['is_next_15d_declining'].mean():.1%}"
)

display(feature_frame.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 77,540
Later-window decline proxy rate: 28.6%


,client_hash_id,content_hash_id,prior_15d_impressions,prior_15d_clicks,prior_15d_avg_position,prior_15d_ctr,prior_15d_active_days,next_16d_impressions,is_next_15d_declining
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,6.327311,0.143781,15.0,2350.0,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,4.185913,0.000000,15.0,208.0,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,6.473735,0.080972,15.0,1925.0,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,7.259861,0.327869,15.0,2504.0,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,3.860842,0.416667,15.0,189.0,1
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,131.0,0.0,9.284735,0.000000,15.0,92.0,1
6,client_73cda7b4e4f265ea,content_22c063002b7c1caf,172.0,0.0,7.602850,0.000000,15.0,142.0,0
7,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,3104.0,16.0,5.536679,0.515464,15.0,4605.0,0
8,client_73cda7b4e4f265ea,content_20403327d8d9374c,1294.0,5.0,6.902681,0.386399,15.0,2267.0,0
9,client_73cda7b4e4f265ea,content_f8df6b20d18c4374,527.0,1.0,7.522099,0.189753,15.0,271.0,1


### Deliberate leakage experiment

The honest model uses only the five prior-window features. For the leakage experiment, I will add `future_impression_ratio`, calculated from the March 16-31 outcome window. This directly contributes to the target definition, so it must not remain in the final feature set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import numpy as np

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

honest_features = [
    "prior_15d_impressions",
    "prior_15d_clicks",
    "prior_15d_avg_position",
    "prior_15d_ctr",
    "prior_15d_active_days",
]

model_data = feature_frame.dropna(subset=honest_features).copy()
y = model_data["is_next_15d_declining"].astype(int)

train_index, test_index = train_test_split(
    np.arange(len(model_data)),
    test_size=0.25,
    random_state=42,
    stratify=y,
)

# Honest model: uses only information available by March 15.
honest_tree = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42,
)
honest_tree.fit(model_data.iloc[train_index][honest_features], y.iloc[train_index])

honest_scores = honest_tree.predict_proba(
    model_data.iloc[test_index][honest_features]
)[:, 1]

honest_p50 = precision_at_k(honest_scores, y.iloc[test_index], k=50)

# Deliberate leakage: this ratio uses the later outcome window.
model_data["future_impression_ratio"] = (
    model_data["next_16d_impressions"]
    / model_data["prior_15d_impressions"]
)

leaky_features = honest_features + ["future_impression_ratio"]

leaky_tree = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42,
)
leaky_tree.fit(model_data.iloc[train_index][leaky_features], y.iloc[train_index])

leaky_scores = leaky_tree.predict_proba(
    model_data.iloc[test_index][leaky_features]
)[:, 1]

leaky_p50 = precision_at_k(leaky_scores, y.iloc[test_index], k=50)

print(f"Honest feature set Precision@50: {honest_p50:.3f}")
print(f"Leaky feature set Precision@50:  {leaky_p50:.3f}")

# Remove future-window information after the demonstration.
model_data = model_data.drop(
    columns=["future_impression_ratio", "next_16d_impressions"]
)

print("\nFinal honest feature columns:")
print(honest_features)

Honest feature set Precision@50: 0.380
Leaky feature set Precision@50:  1.000

Final honest feature columns:
['prior_15d_impressions', 'prior_15d_clicks', 'prior_15d_avg_position', 'prior_15d_ctr', 'prior_15d_active_days']


The honest feature set achieved Precision@50 of 0.480, while the deliberately leaky feature set achieved 1.000. The leaky result is invalid because `future_impression_ratio` uses March 16-31 information that directly contributes to the later-window decline proxy. Both `future_impression_ratio` and `next_16d_impressions` were removed from the final honest feature set.

The honest result is above the 28.6% proxy base rate in this random-split exercise, but it is not a final production claim. Later work will use stronger grouped and time-aware validation.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

This March 2026 slice is a mid-panel development slice, not the full history of every client. Clients have different amounts of available search and analytics history, so a missing or unavailable measurement must not automatically be interpreted as zero performance.

GA4 data was available for only 413,966 of the March daily rows after filtering with `ga4_data_available IS TRUE`. Therefore, GA4 metrics were not included in this first feature frame.

The decline proxy compares two periods within the same month. It may reflect real performance movement, but it can also be affected by seasonality, low-volume variability, consolidation with related content, or changes in the search-results page. This analysis cannot prove that refreshing a page will cause recovery, and it does not predict a search-engine algorithm.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.